In [1]:
import pandas as pd

ground_truth = pd.read_csv('data/ground_truth-new.csv')
ground_truth = ground_truth.to_dict(orient='records')

In [2]:
from Module01_AgenticRAG.ingest import load_faq_data, built_index
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)
documents = documents_llm
index = built_index(documents)

In [3]:
documents[1]

{'id': '977bf7786c',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}

In [4]:
doc_ids = {}

for doc in documents:
    doc_ids[doc['id']] = doc

In [5]:
q = ground_truth[10]

In [6]:
doc_ids[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_clint = OpenAI(
    api_key=os.getenv('API_KEY'),
    base_url=os.getenv('API_URL'),
)

In [9]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_clint,
    course='llm-zoomcamp',
    instructions=INSTRUCTIONS,
    model='llama-3.3-70b-versatile'
)

In [10]:
answer = assistant.rag_pipeline(q['question'])

Rate limit hit. Sleeping for 4s before retry (Attempt 1/5)...
Rate limit hit. Sleeping for 8s before retry (Attempt 2/5)...
Rate limit hit. Sleeping for 16s before retry (Attempt 3/5)...
Rate limit hit. Sleeping for 32s before retry (Attempt 4/5)...


KeyboardInterrupt: 

In [ ]:
assistant.total_cost()

In [ ]:
print(answer)

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_ids[doc_id]

    answer_llm = assistant.rag_pipeline(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

In [ ]:
assistant.reset_usage()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
import time
import math
from concurrent.futures import ThreadPoolExecutor

ground_truth = [rec for rec in ground_truth if rec["document"] in doc_ids]
print(f"Filtered ground_truth to {len(ground_truth)} records.\n")

# 2. RUN THE EVALUATION:
# Calculate the size of each chunk (split into 20 parts)
num_parts = 5
chunk_size = math.ceil(len(ground_truth) / num_parts)

results = []

# max_workers=1 prevents the RateLimitError
with ThreadPoolExecutor(max_workers=1) as pool:
    for i in range(num_parts):
        # Get the current chunk of data
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, len(ground_truth))
        chunk = ground_truth[start_idx:end_idx]

        # Skip if the chunk is empty
        if not chunk:
            break

        print(f"Processing part {i + 1} of {num_parts} (items {start_idx} to {end_idx - 1})...")

        # Process the current chunk
        chunk_results = map_progress(pool, chunk, generate_rag_answer)
        results.extend(chunk_results)

        # Sleep after each chunk (except the very last one)
        if i < num_parts - 1:
            sleep_time = 10  # Wait 20 seconds to let the TPM bucket reset
            print(f"Sleeping for {sleep_time} seconds to avoid rate limit...")
            time.sleep(sleep_time)

print(f"\nFinished processing! Total results: {len(results)}")

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)